# Lab 15-04 · Feedback Properties and Steady-State Error

Four parts, each built on `steady.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
from steady import system_type, error_constants, ss_errors, simulate_error, is_stable

## Part 1 · Predicted and simulated errors

*Predict:* which entries are infinite for each loop?

In [2]:
loops = {
    "50/((s+2)(s+5))": ([50], np.polymul([1, 2], [1, 5])),
    "10/(s(s+4))": ([10], [1, 4, 0]),
    "4(s+1)/(s^2(s+5))": ([4, 4], np.polymul([1, 0, 0], [1, 5])),
    "20(s+2)/(s(s+4)(s+5))": ([20, 40], np.polymul([1, 0], np.polymul([1, 4], [1, 5]))),
}
for name, (n, d) in loops.items():
    assert is_stable(n, d)
    pred = ss_errors(n, d)
    sim = [simulate_error(n, d, kind, 80)[1][-1] for kind in ("step", "ramp", "parabola")]
    print(f"{name:24s} type {system_type(n, d)}  K = {error_constants(n, d)}")
    print(f"{'':24s} predicted {pred}   simulated at t = 80: {np.round(sim, 4)}")

50/((s+2)(s+5))          type 0  K = (5.0, 0.0, 0.0)
                         predicted (0.16666666666666666, inf, inf)   simulated at t = 80: [1.667000e-01 1.343060e+01 5.411137e+02]


10/(s(s+4))              type 1  K = (inf, 2.5, 0.0)
                         predicted (0.0, 0.4, inf)   simulated at t = 80: [ 0.    0.4  31.94]


4(s+1)/(s^2(s+5))        type 2  K = (inf, inf, 0.8)
                         predicted (0.0, 0.0, 1.25)   simulated at t = 80: [-0.    0.    1.25]


20(s+2)/(s(s+4)(s+5))    type 1  K = (inf, 2.0, 0.0)
                         predicted (0.0, 0.5, inf)   simulated at t = 80: [ 0.     0.5   39.725]


## Part 2 · Where the integrator sits

*Predict:* the steady offset with proportional control and with PI.

In [3]:
def position_loop(ki, t_end=20):
    def rhs(t, x):
        a, y, z = x
        u = 10 * (-y) + ki * z + 0.5        # reference 0, disturbance 0.5 at the plant input
        return [(u - a) / 0.1, a, -y]
    t = np.linspace(0, t_end, 20001)
    sol = solve_ivp(rhs, (0, t_end), [0, 0, 0], t_eval=t, rtol=1e-10, atol=1e-12)
    return t, sol.y[1]
fig, ax = plt.subplots(figsize=(6, 3.2))
for ki, lab in ((0, "proportional"), (20, "PI")):
    t, y = position_loop(ki)
    ax.plot(t, y, label=lab)
    print(f"{lab:12s}: peak {y.max():.4f} rad at {t[np.argmax(y)]:.3f} s, final {y[-1]:.5f} rad")
ax.set_xlim(0, 4); ax.set_xlabel("t (s)"); ax.set_ylabel("angle offset (rad)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "disturbance.cir"], capture_output=True, text=True, timeout=120).stdout
    print("\n".join(l for l in out.splitlines() if l.split("=")[0].strip() in ("yp", "ypi") and l.count("=") == 1))
else:
    print("ngspice not installed: skip (CI runs disturbance.cir through tools/run_spice.py)")

proportional: peak 0.0582 rad at 0.363 s, final 0.05000 rad
PI          : peak 0.0505 rad at 0.300 s, final 0.00000 rad
yp                  =  5.000000e-02
ypi                 =  2.355203e-14
yp = 5.000000e-02
ypi = 2.355203e-14


## Part 3 · Holding speed under load

In [4]:
K, R, tau, Kp = 0.02, 1.0, 0.025, 0.2
def speed(ki, t_end=0.6):
    def rhs(t, x):
        w, z = x
        v = Kp * (-w) + ki * z
        return [((v - (R / K) * 0.01) / K - w) / tau, -w]
    t = np.linspace(0, t_end, 60001)
    return t, solve_ivp(rhs, (0, t_end), [0, 0], t_eval=t, rtol=1e-10, atol=1e-12).y[0]
fig, ax = plt.subplots(figsize=(6, 3.2))
for ki in (0, 2, 9.75):
    t, w = speed(ki)
    ax.plot(t * 1e3, w, label=f"Ki = {ki}")
    print(f"Ki = {ki:5}: peak dip {-w.min():.3f} rad/s at {1e3 * t[np.argmin(w)]:.2f} ms, final {w[-1]:.4f} rad/s")
ax.set_xlabel("t (ms)"); ax.set_ylabel("speed change (rad/s)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

Ki =     0: peak dip 2.273 rad/s at 304.17 ms, final -2.2727 rad/s
Ki =     2: peak dip 2.133 rad/s at 9.10 ms, final -0.0090 rad/s
Ki =  9.75: peak dip 1.896 rad/s at 6.04 ms, final -0.0000 rad/s


## Part 4 · A chirp and a hum

In [5]:
wn2, RC = 1e11, 5e-6
t = np.linspace(0, 1e-4, 100001)
for name, r in (("frequency step 100 kHz", 2 * math.pi * 1e5 * t), ("chirp 1e10 Hz/s", 2 * math.pi * 1e10 * t**2 / 2)):
    _, e, _ = signal.lsim(([1, 0, 0], [1, wn2 * RC, wn2]), r, t)
    print(f"PLL, {name}: final phase error {e[-1]:.4f} rad, peak {e.max():.3f} rad")
w0 = 2 * math.pi * 60
t = np.linspace(0, 1.0, 200001)
for name, (num, den) in (("C = 100", ([1, 10], [1, 110])),
                          ("C = 100 + resonant", (np.polymul([1, 0, w0**2], [1, 10]), [1, 110, w0**2 + 3e4, 110 * w0**2]))):
    _, y, _ = signal.lsim((num, den), np.sin(w0 * t), t)
    print(f"hum at the output, {name}: amplitude in the last cycle {np.abs(y[t > 1 - 1 / 60]).max():.4f}")

PLL, frequency step 100 kHz: final phase error 0.0000 rad, peak 0.849 rad


PLL, chirp 1e10 Hz/s: final phase error 0.6283 rad, peak 0.639 rad


hum at the output, C = 100: amplitude in the last cycle 0.9603


hum at the output, C = 100 + resonant: amplitude in the last cycle 0.0001


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.